In [5]:
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.decomposition import PCA
from sklearn.metrics import f1_score, accuracy_score


def cross_validation(dataset, model, n_components):
    """
    Executa Stratified K-Fold utilizando:
    StandardScaler -> PCA -> Modelo

    Retorna o F1 médio.
    """

    x = dataset.data
    y = dataset.target

    skf = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    f1_scores = []

    for train_index, test_index in skf.split(x, y):

        X_train = x[train_index]
        X_test = x[test_index]

        y_train = y[train_index]
        y_test = y[test_index]

        # -------------------------
        # NORMALIZAÇÃO
        # -------------------------

        scaler = StandardScaler()

        X_train_normalizado = scaler.fit_transform(X_train)
        X_test_normalizado = scaler.transform(X_test)

        # -------------------------
        # PCA
        # -------------------------

        pca = PCA(
            n_components=n_components
        )

        X_train_pca = pca.fit_transform(
            X_train_normalizado
        )

        X_test_pca = pca.transform(
            X_test_normalizado
        )

        # -------------------------
        # MODELO
        # -------------------------

        model.fit(
            X_train_pca,
            y_train
        )

        y_pred = model.predict(
            X_test_pca
        )

        # -------------------------
        # F1
        # -------------------------

        f1 = f1_score(
            y_test,
            y_pred,
            average="macro"
        ) * 100

        f1_scores.append(f1)

    return np.mean(f1_scores)


def test_components(dataset, model):
    """
    Testa todas as quantidades possíveis
    de componentes do PCA.

    Retorna uma lista de dicionários.
    """

    resultados = []

    quantidade_features = len(
        dataset.feature_names
    )

    for componentes in range(
        1,
        quantidade_features + 1
    ):

        f1 = cross_validation(
            dataset=dataset,
            model=model,
            n_components=componentes
        )

        resultados.append({
            "componentes": componentes,
            "f1": round(f1, 2)
        })

    return resultados


def main():

    # -------------------------
    # DATASET
    # -------------------------

    from sklearn.datasets import load_iris
    from sklearn.ensemble import RandomForestClassifier

    dataset = load_iris()

    # -------------------------
    # MODELO
    # -------------------------

    model = RandomForestClassifier(
        random_state=42
    )

    # -------------------------
    # TESTAR PCA
    # -------------------------

    resultados = test_components(
        dataset=dataset,
        model=model
    )

    # -------------------------
    # RESULTADO
    # -------------------------

    print(top_k(resultados,2))


if __name__ == "__main__":
    main()

[{'componentes': 4, 'f1': np.float64(95.99)}, {'componentes': 3, 'f1': np.float64(93.3)}]


In [2]:
def top_k(resultados, k):
    return sorted(
        resultados,
        key=lambda x: x["f1"],
        reverse=True
    )[:k]


